# Rust 111: Errors in Depth

Chapter 11 of the Rust track, and the applied track's second stop: what happens when your own types fail. Chapter 08 used `Result` as a value to assert on; chapter 09 logged errors that carried context and backtraces via `eyre`; this chapter opens the machinery underneath — error enums you define yourself, the `From` conversions `?` silently performs, the `source()` chain that walks from wrapper to cause, and the process-level contract of `main() -> Result`. The crates from chapter 09 are nowhere in sight: everything here is `std`, and `eyre`/`anyhow` end the chapter as *consequences* of the manual machinery, not magic.

Everything here is a **runnable example**. Read the markdown, run the cell below it, and change nothing — the practice twin of this notebook is `11_errors-in-depth_p.ipynb`.

## Who this is written for

You are assumed to be a competent programmer arriving from Python and C++. You have raised exceptions, returned error codes, and caught both — none of that is new. What is new: **Rust has one error *mechanism* and no error *language*** — a failure is a value of type `Result<T, E>`, and everything else (the `?` operator, the `Error` trait, backtraces) is built out of it in plain sight. Python's exceptions unwind invisibly through frames; C++'s `throw` does the same; Rust's error value travels only where you write `?`, and *what* it is along the way is a type you chose.

## How to read this notebook

- Markdown cells explain; code cells demonstrate. **Run each code cell as you reach it.** Cells are independent — no sandbox, no files touched, so restart-and-run-from-the-top always works.
- Code that intentionally *fails* is shown as a comment, together with the diagnostic or panic message you would get. §9 is the catalogue.
- The `FeedError` / `PipelineError` examples from §1 onward are one running cast; the definitions from §1 stay live for the rest of the notebook.
- The prose is terse and the facts are dense. This is a reference you will come back to, not a novel you read once.

## The one idea to internalise

**An error is a value; the `?` is the only syntax that moves it.** Design the type first — what happened, in the vocabulary of your domain — then make it a `Debug + Display` type that implements `std::error::Error`, and let `?` plus `From` carry it upward through layers that should not know its details. Python's `raise` names a class and Python's `except` catches it wherever; Rust makes each layer name the error type it *emits* and the error type it *returns*, and refuses to compile the gap between them unless you bridge it — which is the chapter's whole point: the gap-filling is where the engineering lives.


### Contents

- [0. Notebook Conventions: How These Cells Run](#0-notebook-conventions-how-these-cells-run)
- [1. The Error Enum: Variants, Display, Error](#1-the-error-enum-variants-display-error)
- [2. From Conversions: The Upward Hop](#2-from-conversions-the-upward-hop)
- [3. Walking the Chain: `source()`](#3-walking-the-chain-source)
- [4. `Box<dyn Error>`: The Catch-All](#4-boxdyn-error-the-catch-all)
- [5. Returning Result from main: The Process Contract](#5-returning-result-from-main-the-process-contract)
- [6. Panics vs Recoverable Errors](#6-panics-vs-recoverable-errors)
- [7. Backtraces in Errors: The Stable Surface](#7-backtraces-in-errors-the-stable-surface)
- [8. The Cost Shape of `Result`](#8-the-cost-shape-of-result)
- [9. Reading the Compiler: E-Codes](#9-reading-the-compiler-e-codes)
- [10. Mental Model: Error Design](#10-mental-model-error-design)
- [11. Cheat Sheet](#11-cheat-sheet)
- [12. Review](#12-review)

If you are returning to this chapter later, §11 is the one-page summary, §9 the diagnostics catalogue, and §8 the cost numbers.


## 0. Notebook Conventions: How These Cells Run

This notebook runs on the **evcxr** kernel, which wraps each cell into a generated program and compiles it incrementally. Three consequences matter for every cell below:

- **There is no `fn main`.** The kernel supplies one and refuses a user-defined one — which is why §5's `main() -> Result` contract is demonstrated *about* `main` (with a nested `run` function standing in for the program body), never *inside* a cell.
- **State persists across cells.** A `let` from cell 4 is still live in cell 20 — including `use` statements.
- **Items are order-independent, statements are not.** Two `fn`s may call each other regardless of order; two `let`s may not use each other's names before they run.

There is **one namespace per notebook**. Two items with the same name collide exactly as they would in one source file, even in different sections. This chapter *defines error types*, so the rule gets sharper: every `enum`, `struct` and `impl` name below is unique across the whole notebook. If you re-run a cell that defines an item, it is redefined; on a redefinition error, restart the kernel and run from the top.

A failing cell does not poison the kernel — the deliberate compile errors in this chapter are comments, so every cell still runs.

### A chapter without a sandbox

Like chapter 10, this notebook touches **no files** and uses **no crates**. Errors are values; making and moving them is in-process by nature. The one tool from chapter 09 that reappears is `Backtrace` — but from `std`, not from `eyre` (§7).


In [ ]:
// 0.1 Smoke test: build, return, inspect — the chapter in one line.
let qty: Result<u32, String> = Ok(300);
let rejected: Result<u32, String> = Err("qty must be positive".to_string());
println!("ok = {qty:?}, err = {rejected:?}");
assert_eq!(qty, Ok(300));
assert!(rejected.is_err());


## 1. The Error Enum: Variants, Display, Error

An error type is just a type. The convention — the shape every serious Rust API settles on — is an **enum whose variants name the failure modes** your function can actually produce, with each variant carrying exactly the data a caller would need:

| Ingredient | Status | Written as |
|---|---|---|
| `#[derive(Debug)]` | **required** | `Error` demands it (E0277 without) |
| `impl Display` | **required by convention** | the human headline, `write!`-built |
| `impl std::error::Error` | **required by convention** | unlocks `source()`, `Box<dyn Error>` |
| `Clone`/`PartialEq` | optional | only if tests want to assert on the error |

The chapter 09 shortcut is worth naming before it becomes habit: `eyre!` and `wrap_err!` build chains of *messages*. A message is a fine top layer — but a type is what a caller can `match` on, and a `match` on an error enum is a *contract*: add a variant and every `match` without a wildcard arm fails to compile. Exceptions name a class after the fact; Rust makes the failure modes a compile-checked list.

### The cast, defined


In [ ]:
// 1.1 The manual error enum: variants, Display, and the empty Error impl.
use std::fmt::{self, Display};

#[derive(Debug)]
enum FeedError {
    /// The socket dropped mid-session; retries spent, symbol lost with it.
    Disconnected { symbol: String, attempts: u32 },
    /// A quote arrived that could not be parsed; keep the raw text.
    BadQuote { raw: String },
    /// The heartbeat did not arrive within the deadline.
    Timeout(u32),
}
impl Display for FeedError {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        match self {
            FeedError::Disconnected { symbol, attempts } => {
                write!(f, "feed disconnected for {symbol} after {attempts} attempts")
            }
            FeedError::BadQuote { raw } => write!(f, "malformed quote: {raw:?}"),
            FeedError::Timeout(ms) => write!(f, "timeout after {ms} ms"),
        }
    }
}
impl std::error::Error for FeedError {} // no source(): the default returns None

// exercising the three variants:
let errs: Vec<FeedError> = vec![
    FeedError::Disconnected { symbol: "ES".to_string(), attempts: 3 },
    FeedError::BadQuote { raw: "12,x5".to_string() },
    FeedError::Timeout(250),
];
for e in &errs {
    println!("{e}"); // Display: the headline a log line or user sees
}
assert!(format!("{:?}", errs[0]).contains("Disconnected")); // Debug: the shape
assert!(errs[1].to_string().starts_with("malformed quote")); // to_string uses Display


Three reads on that cell:

- **Display is the headline, Debug is the shape.** `feed disconnected for ES after 3 attempts` is what a human reads; `Disconnected { symbol: "ES", attempts: 3 }` is what a log parser and a developer want. This is the same two-audiences split as §6 of chapter 09 — `eyre`'s chain prints Display per hop and Debug for the whole chain.
- **The `Error` impl is empty on purpose**: `impl std::error::Error for FeedError {}`. The trait has two jobs — demand `Debug + Display` (as *supertraits*) and provide `source()` — and with no cause to expose, the default `source()` returning `None` is correct. An empty body is a fact, not a stub.
- **What a variant carries is an API decision.** `BadQuote { raw }` keeps the offending text because an operator will ask "which line was malformed?"; `Timeout(u32)` carries the deadline because a retry policy needs it. Neither carries a backtrace — §7 adds that as a *field*, deliberately.

### The trap the derive list avoids: `Error` demands `Debug`

Drop `#[derive(Debug)]` and the trait bound is what fires — with a diagnostic that names the *trait declaration*, not your struct:

```text
error[E0277]: `Plain` doesn't implement `Debug`
  --> src/main.rs:1:10
   |
   | enum Plain {}
   |      ^^^^^ `Plain` cannot be formatted using `{:?}`
   |
   = help: the trait `Debug` is not implemented for `Plain`
note: required by this bound in `std::error::Error`
   |
   | pub trait Error: Debug + Display {}
   |                  ------  ------- required by this bound in `Error`
```

That note is the trait's definition speaking: `pub trait Error: Debug + Display` — implementing it *requires* both supertraits. `Display` would fire the same way if you had skipped it; `Debug` is merely the easier one to forget because a plain error enum often never prints with `{:?}` until a panic does.


### Wrapping: an error becomes another error's *cause*

Layered systems do not invent failures; they *translate* them. The ingest stage does not know about sockets — it knows the pipeline failed at ingest, and it knows *who* told it so. The wrapper pattern: one variant holds the inner error as a field, plus whatever context the layer owns.


In [ ]:
// 1.2 A wrapper error: the cause is a field; the context is the layer's own.
#[derive(Debug)]
enum PipelineError {
    /// A stage failed; we record which stage and keep the cause.
    Feed { source: FeedError, stage: &'static str },
}
impl Display for PipelineError {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        match self {
            PipelineError::Feed { stage, .. } => {
                write!(f, "pipeline failed at stage {stage}")
            }
        }
    }
}
impl std::error::Error for PipelineError {
    fn source(&self) -> Option<&(dyn std::error::Error + 'static)> {
        match self {
            PipelineError::Feed { source, .. } => Some(source),
        }
    }
}
let wrapped = PipelineError::Feed {
    source: FeedError::BadQuote { raw: "12,x5".to_string() },
    stage: "ingest",
};
println!("display : {wrapped}");     // the layer's headline
println!("debug   : {wrapped:?}");   // the whole struct, cause included
println!("to_string: {}", wrapped.to_string()); // Display again


Read the two outputs as two audiences, as chapter 09's chain did: Display says *what layer failed* (`pipeline failed at stage ingest`) and Debug says *everything it knows* (`Feed { source: BadQuote { raw: "12,x5" }, stage: "ingest" }`). The cause is `private` data — only reachable through `source()` (§3) or a downcast (§4) — which is exactly what makes the wrapper's Display headline *stable*: log parsers match on `pipeline failed at stage ...` while the cause can change release to release.

One detail hides in the `match` arms: `PipelineError::Feed { stage, .. }` uses `..` because Display needs only the stage — naming `source` there would only tempt a `write!` that duplicates Debug's job. The impls agree on the division of labour: **Display = headline, Debug = dump, `source()` = traversal**.

## 2. From Conversions: The Upward Hop

`?` is not sugar for `unwrap`; it is sugar for a `match` that **converts**. Written on a `Result<T, Inner>` inside a function returning `Result<U, Outer>`, it expands to:

```rust
match result {
    Ok(v) => v,
    Err(e) => return Err(Outer::from(e)), // the From impl decides what the error becomes
}
```

Two things follow. **`?` early-returns** — the function is over, this level never hears about the `Ok` path again — and **`?` converts, using `From`**. If `Outer: From<Inner>` does not hold, the code does not compile:

```text
error[E0277]: `?` couldn't convert the error to `PipelineError`
   |
   |     r?;
   |     -^ the trait `From<FeedError>` is not implemented for `PipelineError`
   |     |
   |     this can't be annotated with `?` because it has type `Result<_, FeedError>`
   |
   = note: the question mark operation (`?`) implicitly performs a conversion
           on the error value using the `From` trait
```

The diagnostic is the desugaring talking: it tried `From<FeedError> for PipelineError` and there was no such impl. The fix is not `unwrap` and not a `match` — it is the bridge:

### Writing the bridge


In [ ]:
// 2.1 The From bridge that ? rides on, then two fallible layers composed with ?.
impl From<FeedError> for PipelineError {
    fn from(source: FeedError) -> Self {
        // the conversion is also where the context gets stamped:
        PipelineError::Feed { source, stage: "ingest" }
    }
}
/// Leaf: parses a quantity, failing as FeedError.
fn parse_qty(raw: &str) -> Result<u32, FeedError> {
    raw.parse::<u32>().map_err(|_| FeedError::BadQuote { raw: raw.to_string() })
}
/// Leaf: pretends to connect; fails as FeedError below 3 attempts.
fn fetch(attempts: u32) -> Result<(), FeedError> {
    if attempts < 3 {
        Err(FeedError::Disconnected { symbol: "ES".to_string(), attempts })
    } else {
        Ok(())
    }
}
/// Layer: returns PipelineError — but ? on FeedError results just works.
fn ingest(raw: &str) -> Result<u32, PipelineError> {
    let qty = parse_qty(raw)?; // FeedError -> PipelineError via the bridge
    fetch(3)?;                 // same bridge; stage stamped in `from`
    Ok(qty)
}
// happy path first:
let qty = ingest("300").expect("300 parses");
println!("ingested {qty}");
// error path: what ? built at each hop:
let err = ingest("abc").unwrap_err();
println!("{err}"); // Display of the WRAPPER — the ? converted, not replaced
assert!(matches!(err, PipelineError::Feed { stage: "ingest", .. }));


Two details worth pausing on. **The conversion is where context is stamped** — `from` receives the bare `FeedError` and returns it wrapped with `stage: "ingest"`, so *every* `?` from this layer gets the stage for free; callers cannot forget the context, and the leaf cannot even know it. And **the failure stays a `PipelineError`**, not a `FeedError`: `?` converted it at the boundary, which is why `matches!(err, PipelineError::Feed { .. })` holds. A layer that leaks a foreign error type through its return type has broken its own signature's promise — the compiler makes that impossible rather than unlikely.

`map_err` deserves its own cell rather than one line of contrast: it is the *manual* conversion — the same expansion point as `?`, but your closure decides what the error becomes — useful exactly when the blanket stamp of `From` is not what every callsite wants.

### `map_err`: the per-callsite conversion

`?` is overloaded on `Option` too — inside a function returning `Option<T>`, `opt?` is `match opt { Some(v) => v, None => return None }`. No conversion, no `From`: the early return *is* the whole story.


In [ ]:
// 2.2 map_err: the per-callsite conversion when one From stamp is not enough.
fn validate_raw(raw: &str, attempts: u32) -> Result<u32, PipelineError> {
    let qty = parse_qty(raw).map_err(|e| PipelineError::Feed { source: e, stage: "ingest" })?;
    fetch(attempts).map_err(|e| PipelineError::Feed { source: e, stage: "handshake" })?;
    Ok(qty)
}
let a = validate_raw("abc", 3).unwrap_err(); // parse fails: stamped "ingest"
let b = validate_raw("300", 2).unwrap_err(); // handshake fails: stamped "handshake"
println!("{a} / {b}");
assert!(matches!(a, PipelineError::Feed { stage: "ingest", .. }));
assert!(matches!(b, PipelineError::Feed { stage: "handshake", .. }));
// the same two leaves went through the same wrapper - only the stamps differ


One `From` impl exists per *type pair*; there are as many `map_err` stamps as there are callsites that disagree with it. The rule of thumb: reach for `From` when one conversion is right for every `?`, and `map_err` when a callsite needs its own context (or wants to drop or log the cause). Everything else about the hop — early return, `Ok` unwrapping — is identical.

### `?` on `Option`: the other half of the operator

`?` is overloaded on `Option` too — inside a function returning `Option<T>`, `opt?` is `match opt { Some(v) => v, None => return None }`. No conversion, no `From`: the early return *is* the whole story.


In [ ]:
// 2.3 ? on Option inside an Option-returning function; ok_or is the one free mixed hop.
/// Half the first price in the candle window; None if the window is empty.
fn first_px(candles: &[f64]) -> Option<f64> {
    let first = *candles.first()?; // None -> return None
    Some(first / 2.0)
}
assert_eq!(first_px(&[4126.5, 4127.0]), Some(2063.25));
assert_eq!(first_px(&[]), None); // ? fired and the function returned None
println!("first_px(&[]) = {:?}", first_px(&[]));
// the one mixed hop std gives you: Option -> Result via ok_or
// (the reverse is deliberately missing: an error cannot become "no answer")
let parsed: Option<u32> = "7".parse::<u32>().ok();
assert_eq!(parsed.ok_or("not a number"), Ok(7));
assert_eq!(None::<u32>.ok_or("not a number"), Err("not a number"));


The load-bearing line in the spec of `?` is *same-shape in, same-shape out*: `?` on `Result` needs a `Result`-returning function, `?` on `Option` needs an `Option`-returning one — and the only *mixed* hop std offers for free is `Option` → `Result` (`ok_or`), never the reverse: an error cannot honestly become "no answer". That asymmetry is a design stance — chapter 05's `Option`/`Result` split was exactly this: absence of a value is not a failure, and Rust refuses to blur them even inside one operator.

## 3. Walking the Chain: `source()`

A wrapped error is a linked list: each node's `source()` returns its cause, and the tail — an error with no cause — returns `None`. The trait's default is `None`, so a chain costs one `fn source` per *wrapping* layer and nothing at the leaves. Display shows one node; Debug shows the nodes inline; **`source()` is how you traverse without either**:

### The chain walk


In [ ]:
// 3.1 Walking the chain: Display is one node; source() is the whole list.
let err = ingest("abc").unwrap_err(); // Feed { source: BadQuote { .. }, stage }
println!("node 0 (Display): {err}");
let mut chain: Vec<String> = vec![format!("{err}")];
let mut cause = std::error::Error::source(&err);
while let Some(e) = cause {
    chain.push(format!("{e}"));
    cause = e.source(); // the next hop: same method, one node down
}
for (i, node) in chain.iter().enumerate() {
    println!("chain[{i}]: {node}");
}
assert_eq!(chain.len(), 2); // wrapper + leaf
assert!(chain[1].starts_with("malformed quote"));


Two subtleties, both the compiler's doing:

- **The call needs the fully qualified form** `std::error::Error::source(&err)` on the *first* hop — not because `source` is rare, but because `err`'s concrete type is `PipelineError`, and method resolution would happily find... nothing: `source` exists only *through* the trait, and while one `use std::error::Error;` would fix it, the explicit form says what it does. From the second hop on, `e.source()` is an unambiguous trait-object call.
- **`cause` is a `&dyn Error` in transit** — the walk is ordinary linked-list iteration over trait objects, and chapter 08's `dyn` rules all apply: the type behind the pointer is erased, all you can do is call `Error`'s methods — `source()`, `to_string()` — or downcast (§4).

The chain in that cell is what `eyre`'s `Caused by:` block *prints*: `0: pipeline failed at stage ingest` / `1: malformed quote: "abc"` — chapter 09's numbered list was this loop with formatting. Layered logging (09e §6) should record the *chain*, not just the headline: the `error = %report` field prints the outermost node's Display, and the chain itself is the report's `Debug` — the `Caused by:` block 6.1 showed.

### `source()` vs a public `cause` field: the API question

Nothing stops you from adding `pub fn cause(&self) -> &FeedError` to `PipelineError` — the *typed* accessor. The difference is contract width:

| | `fn source()` (trait) | `pub fn cause()` (inherent) |
|---|---|---|
| returns | `&(dyn Error + 'static)` — type erased | `&FeedError` — typed |
| reachable by | generic code holding any `&dyn Error` | code that knows the concrete type |
| right for | logging, chains, `Box<dyn Error>` plumbing | callers that genuinely need the payload |

The rule of thumb: **the trait method is for the chain, the typed accessor is for the payload.** If callers `match` on the cause — and ours do — give them the typed accessor too; `source()` alone would force every such caller back through a downcast.

### The typed accessor, built


In [ ]:
// 3.2 The typed accessor: the payload path for callers that know the type.
impl PipelineError {
    /// Typed view of the cause: no dyn, no downcast, full exhaustiveness.
    fn cause(&self) -> &FeedError {
        match self {
            PipelineError::Feed { source, .. } => source,
        }
    }
    /// The layer's own context, readable without touching the cause.
    fn stage(&self) -> &'static str {
        match self {
            PipelineError::Feed { stage, .. } => stage,
        }
    }
}
let err = ingest("abc").unwrap_err();
match err.cause() {
    FeedError::BadQuote { raw } => println!("bad quote text: {raw:?}"),
    other => println!("transport problem: {other}"),
}
assert_eq!(err.stage(), "ingest");


The typed accessor is three lines because the cause is private data behind a `match` — and its return type *is* the contract. Callers that know `PipelineError` get `match err.cause()` with compile-checked arms; callers that only know `dyn Error` get `source()` and downcasts. Both live on one type without competing: `cause()` for the payload, `source()` for the chain.


## 4. `Box<dyn Error>`: The Catch-All

Somewhere below the domain errors sits the **catch-all**: `Box<dyn Error>`. A heap-allocated trait object that accepts *any* error type — because std ships `impl<E: Error + 'static> From<E> for Box<dyn Error>`, the one `From` impl that makes `?` work from *every* leaf error into the box, with no bridge of your own. It is the error type of scripts, `main`s, and tests; of code whose layers are too many to name; of "I do not care *which* thing failed, only that it did":

| | domain enum | `Box<dyn Error>` |
|---|---|---|
| what a caller can do | `match` variants — the contract | `to_string()`, `source()`, downcast |
| cost per error | fixed, inline (§8) | one heap allocation |
| right for | library APIs, hot paths | `main`, scripts, tests, glue |

The cost line matters for this track: every boxed error is an allocation and an indirect call on the way down. Scripts and `main`s pay it happily; a per-tick path in a market-data service should not.

### The catch-all, exercised


In [ ]:
// 4.1 Box<dyn Error>: ? from any error type, downcasts back, stringified.
use std::error::Error;
/// Glue layer: too small to have a domain error of its own.
fn boxed_ingest(raw: &str) -> Result<u32, Box<dyn Error>> {
    let qty = parse_qty(raw)?; // FeedError -> Box<dyn Error>: std's From does it
    Ok(qty)
}
let ok = boxed_ingest("42").expect("parses");
println!("boxed ingest ok: {ok}");
let bad = boxed_ingest("x").unwrap_err();
println!("boxed err display: {bad}");        // Display through the box
println!("boxed err debug  : {bad:?}");      // Debug through the box
// downcast: the Any machinery of chapter 08, pointed at errors
let back = bad.downcast_ref::<FeedError>();
assert!(matches!(back, Some(FeedError::BadQuote { .. })));
let text = bad.to_string(); // Display via the vtable
assert!(text.starts_with("malformed quote"));
// from a plain message — the chain-of-strings idiom:
let msg: Box<dyn Error> = "bad config".into();
println!("from &str: {msg}");
let s_err: Box<dyn Error> = String::from("oops").into(); // String is an Error too
println!("from String: {s_err}");


Four moves in that cell, one per line group:

- **`?` with no bridge of our own** — `parse_qty(raw)?` converts `FeedError` into `Box<dyn Error>` through std's blanket `From`, the same machinery §2 taught, just pre-supplied. This is *the* reason `Box<dyn Error>` feels effortless: the blanket impl is the bridge.
- **`downcast_ref::<FeedError>()`** — the type erasure of `dyn` runs both ways: chapter 08's `&*boxed` / `downcast_ref` pair, with `Error` as the trait object. `downcast_mut` exists too — an error you can *repair* in place before it reaches its final handler.
- **`"bad config".into()`** — `&str` and `String` both implement `Error` (in `std`), so the box accepts bare messages. This is what `eyre!`/`anyhow!` *are* at the bottom: message-shaped errors in a box.
- **`to_string()` through the box** — Display via the vtable; the headline survives erasure.

The trap to carry forward is **what the box cannot do**: no `match` — the variants are erased, so the only way back to `FeedError`'s arms is a downcast, and downcasts are `Option`s, not compile-checked exhaustiveness. That is the price of the catch-all, and it is why §10's decision table puts it at `main` and glue, not in library signatures.

### The chain, through the box


In [ ]:
// 4.2 The chain walk of 3.1 runs identically on erased errors: source() is a trait method.
let layered: Box<dyn Error> = PipelineError::Feed {
    source: FeedError::Timeout(250),
    stage: "normalize",
}.into();
println!("outer: {layered}"); // Display through the vtable
let mut hop = layered.source(); // &dyn Error, one node down
let mut hops = 1;
while let Some(e) = hop {
    println!("  cause: {e}");
    hops += 1;
    hop = e.source();
}
assert_eq!(hops, 2); // wrapper + leaf, type invisible the whole way
assert!(layered.downcast_ref::<PipelineError>().is_some()); // ch08's Any, again


The chain walk of §3.1 runs unchanged through the box — `source()` is a trait method, so the vtable serves it no matter which concrete error hides behind the pointer. This is the mechanism `main() -> Result<(), Box<dyn Error>>` relies on: whatever the layers below chose, the top of the program can still print the whole chain, node by node, without naming a single type.


## 5. Returning Result from main: The Process Contract

A binary does not need to handle its top-level error — it needs to *report* it and exit. `fn main() -> Result<(), E>` is the language's concession to that need, and `E` can be **any type implementing `Debug`** — `String`, your `PriceError`, `Box<dyn Error>`. What the runtime does with the `Err` is fixed and probe-verified on this toolchain:

| Path | stderr prints | process exits |
|---|---|---|
| `Ok(())` | nothing | `0` |
| `Err(e)` | `Error: <e's *Debug*>` | **`1`** |

The Debug choice is deliberate and slightly infamous: `Err("boom".to_string())` prints `Error: "boom"` — **quotes included**, because Debug of a `String` is the quoted form. Domain errors print their derived Debug too: `Error: PriceError(9)`. If you want the Display headline on the way out, print it yourself — `eprintln!("error: {e}")` then `std::process::exit(1)` — but then you own the exit code, which is exactly the trade `main() -> Result` was designed to spare you.

The *type* rule is stricter than the runtime: the body must end with a `Result` on **every** path. A function whose last statement is a `println!` (type `()`) where the signature promises `Result` fails to compile:

```text
error[E0308]: mismatched types
   |
   | fn main() -> Result<(), String> {
   |               ^^^^^^^^^^^^^^^^^ expected `Result<(), String>`, found `()`
   |
   = note: expected enum `Result<(), String>`
           found unit type `()`
```

### The contract, demonstrated about `main`


In [ ]:
// 5.1 The main() -> Result contract, acted out by a nested `run` + explicit contract.
// (fmt/Display in scope from 1.1; Error from 4.1 — one namespace, imports persist.)
#[derive(Debug)]
struct PriceError(u32);
impl Display for PriceError {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        write!(f, "price {} out of range", self.0)
    }
}
impl Error for PriceError {}
/// The whole program body: every fallible hop is `?`.
fn run(px: u32) -> Result<(), PriceError> {
    if px == 0 || px > 10_000 {
        Err(PriceError(px)) // the Err path: this is what main would print
    } else {
        Ok(()) // every path must END with a Result - the E0308 rule
    }
}
// act out both paths of main() -> Result<(), PriceError>:
for px in [4_125u32, 0] {
    match run(px) {
        Ok(()) => println!("px {px}: ok (exit 0 path)"),
        Err(e) => println!("px {px}: Error: {e:?} (exit 1 path)"), // Debug, exactly as Termination prints it
    }
}


The `Error: PriceError(0)` line above is byte-for-byte what a real binary's stderr carries when `main` returns the `Err` — quotes only appear when the error type is `String`, whose Debug is quoted. Scripts should also know the **manual** exit, which needs no `Result` at all: `std::process::exit(3)` ends the process *immediately* — destructors of locals on the stack do not run — with the code you chose. That instant-exit fact pairs with §6's panic path: between `Ok(())` (exit 0), `Err` (exit 1) and `panic!` (exit 101), a Unix-style process has a small, useful error-code vocabulary without any logging at all.

One boundary note for the kernel: none of this runs *inside* a cell — evcxr supplies its own `main`, which returns `()`, and refuses a user-defined one (§0). The contract is learned here, but it bites only in `cargo` binaries — which is where chapter 13 will put it.

## 6. Panics vs Recoverable Errors

A panic is not a big error; it is a **different mechanism**. `Result` is a value in a return channel — typed, matchable, composable; a panic is the *unwind* channel — it tears down frames until someone catches it or the process dies. The chapter 09 framing holds: `Result` errors are the story you design; panics are the ones you did not anticipate. But Rust lets you *catch* unwinding too, which is what makes the two mechanisms meet:

| | `Result` error | panic |
|---|---|---|
| channel | the return value | stack unwind |
| who decides what happens | every `match`/`?` on the way | `catch_unwind`, the panic hook, or nothing |
| default outcome | whatever the caller wants | thread ends; `main` exits 101 |
| payload type | `E` — the type you chose | `Box<dyn Any + Send>` — usually `String` or `&'static str` |
| right for | expected, recoverable failures | bugs: broken invariants, unreachable states |

### The two channels, exercised


In [ ]:
// 6.1 catch_unwind: a panic is catchable, and the process continues.
use std::panic;
let caught = panic::catch_unwind(|| panic!("hard stop"));
assert!(caught.is_err()); // the panic came back as Err(payload)
let payload = caught.unwrap_err();
// the payload of panic!("literal") is a &'static str; formatted panics box a String:
let as_str = payload.downcast_ref::<&'static str>().copied();
let as_string = payload.downcast_ref::<String>().map(String::as_str);
println!("payload: {:?} / {:?}", as_str, as_string);
println!("...and the process is still running"); // unwind != death
// the chapter-08 machinery, pointed at payloads: it is Box<dyn Any + Send>
let p2 = panic::catch_unwind(|| assert_eq!(1, 2)).unwrap_err();
let s2 = p2.downcast_ref::<String>().map(String::as_str);
assert!(s2.unwrap().starts_with("assertion `left == right` failed"));
println!("assert_eq! payload: String");


The payload rules, probe-verified: `panic!("literal")` boxes a `&'static str`; `panic!("qty {x} rejected")` — any *formatted* panic — boxes a `String`; `assert_eq!`/`assert!` box a `String` carrying the assertion text; and a custom error passed through `panic!("{}", err)` arrives as the *formatted `String`*, not as your error type — the panic channel does not preserve your type, only its story. (The `panicked at` lines the cells above emit are the default panic hook doing its job *before* `catch_unwind` receives the payload — catching does not silence the hook.) That last line is the design argument in miniature: **if a caller might want to recover by type, it is a `Result` error, not a panic.** Python's `except PriceError` has no Rust analogue on the panic channel — `catch_unwind` returns `Box<dyn Any>`, and downcasting a `String` back into a `PriceError` is guessing, not recovery.

Two boundary rules finish the section. **The hook is not the catch**: `panic::set_hook` observes every panic (chapter 09 used it for the panic-side log line) but cannot stop the unwind; only `catch_unwind` resumes. And **catching is not always allowed** — a `panic = "abort"` profile turns unwinding into instant process death (§5's `exit`-style no-cleanup exit), which is a deliberate deployment choice for services that would rather die than continue from a torn invariant. Between the two, the chapter's habit: `Result` for everything a caller should *decide* about; panics for everything the caller should *never have to* decide, because it means the code is wrong.

### The hook: observing every panic, catching none


In [ ]:
// 6.2 A panic hook: sees every panic before any catch_unwind; cannot stop the unwind.
// (take/set: evcxr installs its own hook - put it back when the demo ends.)
use std::sync::atomic::{AtomicUsize, Ordering};
static PANICS: AtomicUsize = AtomicUsize::new(0);
let previous = panic::take_hook();
panic::set_hook(Box::new(|info| {
    PANICS.fetch_add(1, Ordering::Relaxed);
    println!("[hook] observed: {info}");
}));
let _ = panic::catch_unwind(|| panic!("observed, then caught"));
panic::set_hook(previous);
let seen = PANICS.load(Ordering::Relaxed);
println!("hook saw {seen} panic(s); process alive");
assert_eq!(seen, 1);


The hook ran *before* `catch_unwind` got its `Err` — the sequence for every panic is hook, then unwind, then (optionally) catch. In a real service the hook is where chapter 09's panic-path logging lives — the log line and the `Backtrace::force_capture()` — precisely because it fires even for panics nobody catches. The take/set pair matters inside evcxr: the kernel already installed a hook that prints the `panicked at` line, so the demo restores it and keeps the notebook's output tidy across re-runs.

A note on the `AtomicUsize`: state shared between the hook and ordinary code must be `Sync`-safe because the hook can fire on any thread — the static-plus-atomic shape is the standard one, and chapter 16 owns the full tour of atomics.

## 7. Backtraces in Errors: The Stable Surface

Chapter 09 captured backtraces *next to* errors (`eyre` reports, `Backtrace::force_capture()` in the panic hook). This chapter pins what is **stable in `std`** for storing one *inside* your error — and what is not:

| Stable, usable today | Nightly on 1.98 (E0658) |
|---|---|
| `Backtrace::capture()` / `force_capture()` as a **field** of your error | `Error::provide` / `request_ref` — the provider API |
| `BacktraceStatus::{Captured, Disabled, Unsupported}` | `error_generic_member_access` (issue #99301) |
| `impl Error` with no `source()` — chain ends | `Backtrace` access through `dyn Error` |
| reading `err.bt.status()` on the concrete type | |

The stable shape is the honest one: **capture at construction, read on the concrete type**. The provider API's promise — `request_ref::<Backtrace>()` walking a `dyn Error` chain without downcasting — is what `eyre` builds its backtrace-on-`Report` story on, and it is why the notebook teaches the *field* pattern: it is the shape that will survive, and it works today.

### The backtrace field, exercised


In [ ]:
// 7.1 A Backtrace field in the error: capture at construction, read later.
use std::backtrace::{Backtrace, BacktraceStatus};
#[derive(Debug)]
struct SnapshotError {
    msg: &'static str,
    bt: Backtrace,
}
impl Display for SnapshotError {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        write!(f, "snapshot failed: {}", self.msg)
    }
}
impl Error for SnapshotError {}
let err = SnapshotError { msg: "disk full", bt: Backtrace::capture() };
println!("display: {err}");
let status = err.bt.status();
println!("bt status: {status:?}");
assert!(matches!(status, BacktraceStatus::Disabled | BacktraceStatus::Captured));
// the gate (chapter 09's cached-first-capture rule) decides which one you get;
// force_capture() ignores the gate and always captures:
let forced = SnapshotError { msg: "disk full", bt: Backtrace::force_capture() };
assert!(matches!(forced.bt.status(), BacktraceStatus::Captured));
// and the derived Debug renders the field in place:
let dbg = format!("{forced:?}");
assert!(dbg.contains("SnapshotError"));
assert!(dbg.contains("bt:")); // the backtrace is part of the dump


Three carries from chapter 09, restated as *error-design* facts:

- **The gate is cached at the process's first `capture()`** (09e §6.2) — so a field captured in a kernel session is `Disabled` unless `RUST_BACKTRACE` was set before the session's first capture. The cell above asserts only the disjunction for that reason; the deterministic half is `force_capture()`, which always reports `Captured`.
- **The backtrace is a field, not a superpower** — it costs what any field costs: capture is a stack walk (dozens of frames on this machine — real work), so put it behind the error path, never in a hot `Ok` path. A `Disabled` backtrace is one flag check; a `Captured` one is the walk you paid for.
- **The nightly provider API changes the *access*, not the storage.** With `error_generic_member_access` stabilised, generic code will `request_ref::<Backtrace>()` down a chain; until then, reading the field on the concrete type — or downcasting to it (§4) — is the portable move. `eyre`/`color-eyre` already do their own capture, which is why chapter 09 never needed any of this.

The prose note to carry: **stable Rust's error backtraces are a convention you build**, and the convention is one field + one capture call. Everything fancier is a crate.

### The status matrix, measured


In [ ]:
// 7.2 The three BacktraceStatus values and which constructor produces them.
// (Backtrace and BacktraceStatus already in scope from 7.1.)
let gated = Backtrace::capture();
let forced = Backtrace::force_capture();
println!("capture()       : {:?}", gated.status()); // Disabled or Captured: session history decides
println!("force_capture() : {:?}", forced.status());     // always Captured
println!("forced frames   : {}", format!("{forced}").lines().count());
// the third arm, Unsupported, means no backtrace support in this build at all -
// probed: unreachable on this toolchain, but the enum arm exists for it
assert!(matches!(
    gated.status(),
    BacktraceStatus::Disabled | BacktraceStatus::Captured
));
assert!(matches!(forced.status(), BacktraceStatus::Captured));
assert!(format!("{forced}").contains("main")); // symbols resolve in debug builds


Which status `capture()` reports depends on the session's history — the cached gate of 09e §6.2 — so the cell asserts the disjunction, not an arm. On this toolchain `Unsupported` never appears (the enum arm exists for builds without backtrace support), and `force_capture()` always reports `Captured` with ~40 symbolised frames including `main`. The third state earns its name in a stripped release binary — the 09e caveat about optimised builds applies verbatim.

## 8. The Cost Shape of `Result`

`Result<T, E>` is an enum — the sum type of §9 in chapter 10 — and its size is the sum's size: **max of the variants, plus room for a discriminant**. Probed numbers for this toolchain (`size_of`, all in bytes):

| Type | Size | Why |
|---|---|---|
| `Result<u32, PriceError>` (`PriceError` = 4) | 8 | u32 + u32, no padding needed |
| `Result<u32, ()>` | 8 | same shape; `()` costs nothing but keeps the tag |
| `Option<Result<u32, ()>>` | 8 | **niche through two layers** — `None` lives inside `Result`'s padding |
| `Result<(), PriceError>` | 8 | the error's 4 bytes + discriminant word |
| `Result<u32, Box<dyn Error>>` | 16 | the box is a fat pointer (ch10 §8) |
| `Result<Big, ()>` (`Big` = 32) | 32 | discriminant hides in the padding of the larger variant |
| `Result<&u32, ()>` | 8 | references have a null niche |
| `Option<Result<&u32, ()>>` | 16 | no niche left inside `Result<&u32, ()>` for `None` |

The design lesson is the first table row, not the niche gymnastics: **make the error type small**. A `Result<u32, PriceError>` is 8 bytes — returnable in registers, free to `?`. An error carrying a `String` message, a backtrace, and a cause chain is a different *cost class* — heap allocations at construction, moving by pointer thereafter. Both are legitimate; §10's table says which layer pays which.

### Measured, in the kernel


In [ ]:
// 8.1 The size table, re-measured (size_of is prelude, per chapter 08).
enum Big {
    A(u64, u64, u64),
    B(u64, u64, u64),
}
println!("Result<u32, PriceError>   = {}", size_of::<Result<u32, PriceError>>());
println!("Result<u32, ()>          = {}", size_of::<Result<u32, ()>>());
println!("Option<Result<u32, ()>>  = {}", size_of::<Option<Result<u32, ()>>>());
println!("Result<(), PriceError>   = {}", size_of::<Result<(), PriceError>>());
println!("Result<u32, Box<dyn Error>> = {}", size_of::<Result<u32, Box<dyn Error>>>());
let bigs = [Big::A(1, 2, 3), Big::B(4, 5, 6)];
let mut checksum = 0u64;
for b in &bigs {
    match b {
        Big::A(x, y, z) => checksum += x + y + z, // fields genuinely read:
        Big::B(x, y, z) => checksum += x + y + z, // Debug renders don't count (ch08)
    }
}
println!("Big = {} bytes (checksum {checksum}), Result<Big, ()> = {}",
    size_of::<Big>(),
    size_of::<Result<Big, ()>>());
assert_eq!(size_of::<Result<u32, PriceError>>(), 8);
assert_eq!(size_of::<Option<Result<u32, ()>>>(), 8); // niche through two layers
assert_eq!(size_of::<Result<u32, Box<dyn Error>>>(), 16);
assert_eq!(size_of::<Big>(), 32);
assert_eq!(size_of::<Result<Big, ()>>(), 32); // tag hides in padding
println!("Result<(), PriceError> and Option<Result<&u32, ()>> = {}, {}",
    size_of::<Result<(), PriceError>>(),
    size_of::<Option<Result<&u32, ()>>>());


The niche rows repay a second look because they are free performance *policy*, not trivia: `Option<Result<u32, ()>>` is 8 bytes because `None` fits inside the `Result`'s discriminant space, but `Option<Result<&u32, ()>>` is 16 because a reference's null niche is already *spent* making `Result<&u32, ()>` itself 8 — two niches cannot share one pointer. When an API returns `Option<Result<T, E>>` (parse a row *and* validate it), the size tells you whether the layers cost anything: here the `Option` layer is free over `Result<u32, ()>` and not over the reference form. Chapter 08's niche table (§9) is the same fact pointed at `Option` alone.

For the latency-minded: nothing in this chapter's *machinery* costs per-operation on the happy path — `?` on `Ok` is a branch, the error enum travels in registers when small. The costs arrive with the error: the `String`s in `BadQuote`-style payloads, the boxed catch-all, the captured backtrace. The happy path is free; make sure the error path is *allowed* to be expensive, and keep it off the per-tick code.

## 9. Reading the Compiler: E-Codes

This chapter's diagnostics are the machinery talking: a missing supertrait, a missing bridge, a blanket impl colliding with std's own. Every wording below was captured from this exact toolchain.

| Diagnostic | Trigger | Message (abridged) | Fix |
|---|---|---|---|
| **E0277** | `impl Error` without `Debug` (or `Display`) | `` `Plain` doesn't implement `Debug` `` + ``required by this bound in `std::error::Error` `` | derive `Debug`, impl `Display` (§1.1) |
| **E0277** | `?` without a `From` bridge | `` `?` couldn't convert the error to `PipelineError` `` + ``the trait `From<FeedError>` is not implemented`` | write the `From` impl (§2.1) |
| **E0119** | blanket `impl<E: Error> From<E> for MyError` | ``conflicting implementations of trait `From<PriceError>` for type `PriceError` `` + ``impl<T> From<T> for T`` | concrete `From` impls; the blanket slot is std's |
| **E0308** | `main() -> Result` with a non-`Result` tail | ``expected `Result<(), String>`, found `()` `` | end every path with a `Result` (§5) |
| **E0252** | same name imported twice (`use std::fmt::Display;` ×2) | ``the name `Display` is defined multiple times`` | one `use`, or `{self, Display}`-style grouping |
| **E0658** | `Error::provide` / `request_ref` on stable | ``see issue #99301`` — feature `error_generic_member_access` | stable surface only (§7) |

Two worth dwelling on. **E0119 is the trap of ambition**: the tempting `impl<E: Error> From<E> for MyError` — "accept anything!" — collides with std's reflexive `impl<T> From<T> for T` (every type converts to *itself*), so it is refused before it can break `From`'s coherence. The escape is what `anyhow` ships: a *wrapper* type with the blanket impl, which is exactly the `Box<dyn Error>`/`Report` layer of §4. And **the two E0277s are the same diagnostic doing two jobs** — a missing *supertrait* and a missing *conversion* — because both are "trait bound not satisfied"; read the bound named in the note, not just the code.

### The panic-side payload, for the record


In [ ]:
// 9.1 What unwrap() panics with: the Debug of the error, prefixed by the call site.
let r: Result<u32, PriceError> = Err(PriceError(9));
let payload = panic::catch_unwind(|| r.unwrap()).unwrap_err();
let text = payload.downcast_ref::<String>().map(String::as_str).unwrap_or("?");
println!("unwrap payload: {text}");
assert!(text.contains("called `Result::unwrap()` on an `Err` value"));
assert!(text.contains("PriceError(9)")); // Debug of the error - not its Display
// the lesson: unwrap reports the shape; expect reports the intent:
let r2: Result<u32, PriceError> = Err(PriceError(9));
let payload2 = panic::catch_unwind(move || r2.expect("price validated upstream")).unwrap_err();
let text2 = payload2.downcast_ref::<String>().map(String::as_str).unwrap_or("?");
println!("expect payload: {text2}");
assert!(text2.starts_with("price validated upstream"));


## 10. Mental Model: Error Design

The chapter's decisions, compressed into the order you should make them:

| Question | Answer with | Where |
|---|---|---|
| "What can go wrong here?" | an enum's variants, one per failure mode | §1 |
| "What does the human see?" | `Display` — one headline per variant | §1 |
| "What does the log want?" | `Debug` + `source()` chain | §1, §3 |
| "Which layer does the caller sit in?" | `From` bridges stamp context at boundaries | §2 |
| "Should the caller match, or just report?" | match → domain enum; report → `Box<dyn Error>` | §4 |
| "Is it a failure or a bug?" | failure → `Result`; bug → panic | §6 |
| "Who needs the backtrace?" | capture at construction, read on the concrete type | §7 |
| "What does the error cost?" | small inline enum on hot paths; box or context off them | §8 |


Read it as a loop, top to bottom: *enumerate* the failure modes, *name* them with variants, *dress* them with Display, *connect* them with `From` and `?`, and only then decide who is allowed the catch-all. Python arrives at this design by exception-class inheritance (each layer adds a subclass); C++ by exception hierarchies or error codes; Rust by *composition in a single enum* — the wrapper variant holds the cause, and the chain is data structure, not inheritance. That is the applied-track shape of error handling: not a framework, a type you designed, whose variants are the contract.

## 11. Cheat Sheet

| Task | Code |
|---|---|
| the error type | `#[derive(Debug)] enum E { ... }` |
| the required impls | `impl Display for E { fn fmt(..) }` + `impl std::error::Error for E {}` |
| with a cause | `impl Error { fn source(&self) -> Option<&(dyn Error + 'static)> { Some(&self.inner) } }` |
| the bridge | `impl From<Inner> for Outer { fn from(e: Inner) -> Self { Outer::X { source: e } } }` |
| the hop | `let v = fallible()?;` (converts via `From`, early-returns) |
| manual hop | `.map_err(\|e\| Outer::X { source: e, stage: "y" })?` |
| catch-all | `Result<T, Box<dyn Error>>` (std's blanket `From` does the rest) |
| downcast | `err.downcast_ref::<FeedError>()` / `downcast_mut` |
| message error | `"bad config".into()` → `Box<dyn Error>` |
| program contract | `fn main() -> Result<(), E>` — `Err` prints `Error: {e:?}`, exit 1 |
| manual exit | `std::process::exit(3)` (immediate; no destructors) |
| catch a panic | `panic::catch_unwind(\|\| ..)` → `Err(Box<dyn Any + Send>)` |
| panic payloads | `&'static str` for literals, `String` for formatted |
| backtrace field | `struct E { bt: Backtrace }`, `Backtrace::capture()` at construction |
| sizes | `Result<u32, E4>` = 8; `Box<dyn Error>` error = 16; niche can hide the tag |

## 12. Review

Answer from memory; each one is a §-numbered cell away.

1. Your `impl Error for FeedError {}` has an empty body. What two things is the empty body asserting, and what does the default `source()` return? (§1.1)
2. `?` on a `Result<_, FeedError>` inside a function returning `Result<_, PipelineError>` compiled — what impl made it legal, and where did the `stage` context come from? (§2.1)
3. Why does the chain walk's *first* hop use `std::error::Error::source(&err)` while later hops are plain `e.source()`? (§3.1)
4. `Box<dyn Error>` gives you `?` from *every* error type for free. What exact impl supplies that, and what can a caller *no longer do* once an error is boxed? (§4.1)
5. A binary returns `Err("boom".to_string())` from `main`. What exactly appears on stderr, and why the quotes? What exit code? (§5.1)
6. `panic!("{}", err)` and `Err(err)` look similar at a glance. Which mechanism preserves `err`'s *type*, and what does the panic channel deliver instead? (§6.1)
7. `Backtrace::capture()` in your error's constructor reported `Disabled` in the kernel. What rule from chapter 09 decides that, and which call always reports `Captured`? (§7.1)
8. `Option<Result<u32, ()>>` is 8 bytes but `Option<Result<&u32, ()>>` is 16. What is a niche, and why can two layers not share one? (§8.1)
9. You want `impl<E: Error> From<E> for PriceError`. Which diagnostic fires, which std impl does it conflict with, and what is the sanctioned escape? (§9)
10. `unwrap()` and `expect()` panic with the same error. Whose payload names the *call site's intent*, and whose names the *call itself*? (§9.1)

### What Comes Next

Chapter 11 closes the error story; the next chapters take the pressure off `Result` plumbing and put it on *composition*:

| Chapter | Title | What it settles |
|---|---|---|
| **12** | **Iterators in Depth** | the `Iterator` trait from the author's side — and `try_fold`, the loop that `?` wishes it were |
| 13 | Modules, Crates and Testing | `pub` finally bites: error types exported from modules, and `#[should_panic]` tests for §6 |
| 14 | Smart Pointers and Interior Mutability | `Box` from §4 gets siblings: `Rc`, `RefCell`, and when a shared error cache is worth it |

**Carry forward.** You should now be able to do these without looking anything up:

- define an error enum with `Debug`, `Display` and an `Error` impl (empty or with `source()`)
- wire two error types together with `From` and let `?` do the conversion
- walk a `source()` chain and explain what `eyre`'s `Caused by:` block is printing
- choose between a domain enum and `Box<dyn Error>` per layer, and say what the box costs
- state the `main() -> Result` contract (stderr text, exit code) and the panic/`Result` boundary
- capture a backtrace into an error on stable Rust, and size a `Result` at a glance
